# Symptom Text-to-Feature Mapping Pipeline

This notebook constructs a production-ready preprocessing engine that bridges the gap between natural, unstructured patient symptom descriptions and the structured binary feature space required by downstream disease-prediction classifiers. The system employs three complementary matching strategies -- fuzzy string similarity, TF-IDF cosine retrieval, and dense semantic embeddings -- and fuses their scores into a final ensemble decision. Every model artifact is persisted to `Model/symptom_matcher_artifacts/` for reuse in inference pipelines.

---

## Table of Contents

1. Environment Setup
2. Data Loading and Inspection
3. Data Cleaning and Canonical Symptom Extraction
4. Strategy A - Fuzzy String Matching (RapidFuzz)
5. Strategy B - TF-IDF Cosine Retrieval
6. Strategy C - Sentence Transformer Semantic Search
7. Ensemble Fusion of All Strategies
8. End-to-End Inference Pipeline
9. Evaluation Against Ground-Truth Pairs
10. Artifact Persistence

---

## 1. Environment Setup

The cell below installs packages that may not be present in the active environment. The `sentence-transformers` library provides access to pre-trained transformer encoders; `rapidfuzz` is a highly optimized Cython implementation of Levenshtein-based similarity metrics. `faiss-cpu` enables approximate nearest-neighbour search over dense embedding matrices at scale.

In [ ]:
import subprocess
import sys

packages = [
    "rapidfuzz",
    "sentence-transformers",
    "faiss-cpu",
    "nltk",
]

for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("All packages installed successfully.")

In [ ]:
import os
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from rapidfuzz import process as rf_process, fuzz as rf_fuzz
from sentence_transformers import SentenceTransformer

warnings.filterwarnings("ignore")
np.random.seed(42)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

ARTIFACT_DIR = os.path.join("..", "Model", "symptom_matcher_artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)
DATASET_DIR = os.path.join("..", "Dataset")
print(f"Artifact directory: {os.path.abspath(ARTIFACT_DIR)}")

---

## 2. Data Loading and Inspection

The primary training table encodes 4,920 patient records across 132 binary symptom features and a `prognosis` label covering 41 distinct diseases. The supplementary files provide human-readable disease descriptions, medication lists, dietary guidelines, workout recommendations, and precautions -- all keyed on disease name and used in downstream treatment retrieval. `Symptom-severity.csv` assigns clinical severity weights (1 to 7) to each symptom.

In [ ]:
df_train = pd.read_csv(os.path.join(DATASET_DIR, "Training.csv"))
df_severity = pd.read_csv(os.path.join(DATASET_DIR, "Symptom-severity.csv"))
df_symptoms = pd.read_csv(os.path.join(DATASET_DIR, "symtoms_df.csv"))
df_description = pd.read_csv(os.path.join(DATASET_DIR, "description.csv"))

print("Training dataset shape:", df_train.shape)
print("Unique disease classes:", df_train["prognosis"].nunique())
print("\nSeverity table shape:", df_severity.shape)
df_train.iloc[:5, :8].join(df_train[["prognosis"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

disease_counts = df_train["prognosis"].value_counts()
axes[0].barh(
    disease_counts.index,
    disease_counts.values,
    color=plt.cm.viridis(np.linspace(0.1, 0.9, len(disease_counts))),
)
axes[0].set_title("Sample Count per Disease Class", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Count")
axes[0].invert_yaxis()

severity_vals = df_severity["weight"].value_counts().sort_index()
axes[1].bar(severity_vals.index.astype(str), severity_vals.values, color="steelblue", edgecolor="white")
axes[1].set_title("Distribution of Symptom Severity Weights", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Severity Weight (1 = mild, 7 = severe)")
axes[1].set_ylabel("Number of Symptoms")

plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "01_dataset_overview.png"), dpi=150, bbox_inches="tight")
plt.show()

---

## 3. Data Cleaning and Canonical Symptom Extraction

The 132 column headers of `Training.csv` constitute the canonical symptom vocabulary. Several column names contain typographic artefacts such as leading/trailing whitespace, inconsistent underscores, and embedded spaces (e.g., `foul_smell_of urine`, `dischromic _patches`). Each symptom is also normalized using lemmatization and stopword removal to produce a clean text representation used as the retrieval target across all three matching strategies.

Cleaning steps applied:

- Strip leading and trailing whitespace from all labels.
- Collapse multiple consecutive underscores and whitespace into a single underscore.
- Remove parentheses and special characters.
- Build a severity weight lookup dictionary keyed on cleaned symptom names.

In [ ]:
def clean_symptom_name(name: str) -> str:
    name = name.strip()
    name = re.sub(r"[^a-z0-9_ ]", "", name.lower())
    name = re.sub(r"[\s_]+", "_", name)
    name = name.strip("_")
    return name


raw_symptom_cols = [c for c in df_train.columns if c != "prognosis"]
canonical_symptoms = [clean_symptom_name(s) for s in raw_symptom_cols]
col_rename_map = {raw: clean for raw, clean in zip(raw_symptom_cols, canonical_symptoms)}
df_train.rename(columns=col_rename_map, inplace=True)
df_train["prognosis"] = df_train["prognosis"].str.strip()
symptom_display = [s.replace("_", " ") for s in canonical_symptoms]

df_severity["Symptom"] = df_severity["Symptom"].apply(clean_symptom_name)
severity_lookup = dict(zip(df_severity["Symptom"], df_severity["weight"]))

print(f"Total canonical symptoms: {len(canonical_symptoms)}")
print("\nSample cleaned names:")
for raw, clean in list(col_rename_map.items())[:10]:
    print(f"  {repr(raw)} -> {repr(clean)}")

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))


def normalize_query(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    return " ".join(tokens)


symptom_normalized = [normalize_query(s.replace("_", " ")) for s in canonical_symptoms]

symptom_df = pd.DataFrame({
    "canonical": canonical_symptoms,
    "display": symptom_display,
    "normalized": symptom_normalized,
    "severity": [severity_lookup.get(s, 3) for s in canonical_symptoms],
})

print(symptom_df.head(10).to_string(index=False))

---

## 4. Strategy A - Fuzzy String Matching (RapidFuzz)

Fuzzy matching operates purely at the character level using token-based similarity ratios. This approach is highly effective for catching typographic errors, abbreviated terms, and colloquial shortenings. Two RapidFuzz scorers are combined:

- **WRatio**: A weighted composite of multiple Levenshtein variants, robust across different input lengths.
- **token_set_ratio**: Insensitive to word order, critical for multi-word symptoms.

The final fuzzy score for a candidate is `max(WRatio, token_set_ratio) / 100.0`.

In [ ]:
FUZZY_THRESHOLD = 0.60


def fuzzy_match(query: str, top_k: int = 5):
    q_norm = normalize_query(query)
    wratio_scores = rf_process.extract(
        q_norm, symptom_normalized, scorer=rf_fuzz.WRatio, limit=top_k * 2
    )
    tsr_scores = rf_process.extract(
        q_norm, symptom_normalized, scorer=rf_fuzz.token_set_ratio, limit=top_k * 2
    )
    score_map = {}
    for match, score, idx in wratio_scores:
        score_map[idx] = max(score_map.get(idx, 0), score / 100.0)
    for match, score, idx in tsr_scores:
        score_map[idx] = max(score_map.get(idx, 0), score / 100.0)
    ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
    return [
        {
            "canonical": canonical_symptoms[idx],
            "display": symptom_display[idx],
            "score": round(score, 4),
            "severity": symptom_df.loc[idx, "severity"],
        }
        for idx, score in ranked[:top_k] if score >= FUZZY_THRESHOLD
    ]


test_queries = [
    "stomch ache and vomting",
    "my skin is really itchy and has red spots",
    "i feel dizzy and my head spins",
    "difficulty breathing and tight chest",
    "blured vision and headche",
]

print("Fuzzy Matching Results:\n")
for q in test_queries:
    matches = fuzzy_match(q)
    print(f"Query: {repr(q)}")
    for m in matches:
        print(f"  -> {m['display']} (score={m['score']}, severity={m['severity']})")
    print()

---

## 5. Strategy B - TF-IDF Cosine Retrieval

TF-IDF vectorization converts each symptom's normalized text representation into a sparse numeric vector. User queries undergo the same normalization and are projected into the same vector space. A character n-gram analyzer (2-4 grams) is used, making the representation robust to minor spelling variations without requiring character-level fuzzy alignment.

In [ ]:
TFIDF_THRESHOLD = 0.20

tfidf_vectorizer = TfidfVectorizer(
    analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True, min_df=1
)
tfidf_matrix = tfidf_vectorizer.fit_transform(symptom_normalized)

print(f"TF-IDF vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Symptom matrix shape: {tfidf_matrix.shape}")

In [ ]:
def tfidf_match(query: str, top_k: int = 5):
    q_norm = normalize_query(query)
    q_vec = tfidf_vectorizer.transform([q_norm])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = sims.argsort()[::-1][:top_k]
    return [
        {
            "canonical": canonical_symptoms[idx],
            "display": symptom_display[idx],
            "score": round(float(sims[idx]), 4),
            "severity": symptom_df.loc[idx, "severity"],
        }
        for idx in top_indices if sims[idx] >= TFIDF_THRESHOLD
    ]


print("TF-IDF Cosine Retrieval Results:\n")
for q in test_queries:
    matches = tfidf_match(q)
    print(f"Query: {repr(q)}")
    for m in matches:
        print(f"  -> {m['display']} (score={m['score']}, severity={m['severity']})")
    print()

---

## 6. Strategy C - Sentence Transformer Semantic Search

Dense sentence embeddings produced by a pre-trained transformer model capture semantic meaning beyond surface-level character patterns. The model `all-MiniLM-L6-v2` is used: lightweight (22M parameters), CPU-friendly, and producing 384-dimensional embeddings. This allows the matcher to bridge vocabulary gaps such as mapping "breathing difficulty" to "breathlessness".

The symptom corpus is embedded once and cached to disk via NumPy. Subsequent runs load the pre-computed embeddings, avoiding repeated inference costs.

In [ ]:
SBERT_THRESHOLD = 0.45
SBERT_MODEL_NAME = "all-MiniLM-L6-v2"

sbert_model = SentenceTransformer(SBERT_MODEL_NAME)

sbert_corpus_path = os.path.join(ARTIFACT_DIR, "sbert_corpus_embeddings.npy")

if os.path.exists(sbert_corpus_path):
    corpus_embeddings = np.load(sbert_corpus_path)
    print("Loaded pre-computed corpus embeddings from disk.")
else:
    corpus_embeddings = sbert_model.encode(
        symptom_display, normalize_embeddings=True, show_progress_bar=True, batch_size=64
    )
    np.save(sbert_corpus_path, corpus_embeddings)
    print("Corpus embeddings computed and saved.")

print(f"Corpus embedding shape: {corpus_embeddings.shape}")

In [ ]:
def sbert_match(query: str, top_k: int = 5):
    q_emb = sbert_model.encode([query], normalize_embeddings=True)
    sims = (q_emb @ corpus_embeddings.T).flatten()
    top_indices = sims.argsort()[::-1][:top_k]
    return [
        {
            "canonical": canonical_symptoms[idx],
            "display": symptom_display[idx],
            "score": round(float(sims[idx]), 4),
            "severity": symptom_df.loc[idx, "severity"],
        }
        for idx in top_indices if sims[idx] >= SBERT_THRESHOLD
    ]


print("Sentence Transformer Semantic Search Results:\n")
for q in test_queries:
    matches = sbert_match(q)
    print(f"Query: {repr(q)}")
    for m in matches:
        print(f"  -> {m['display']} (score={m['score']}, severity={m['severity']})")
    print()

---

## 7. Ensemble Fusion of All Strategies

The ensemble fuses all three strategies by computing a weighted linear combination of normalized scores:

```
ensemble_score = w_fuzzy * score_fuzzy + w_tfidf * score_tfidf + w_sbert * score_sbert
```

Weights favour SBERT (0.50) for semantic fidelity, with fuzzy (0.30) and TF-IDF (0.20) providing complementary coverage. A small severity weight bonus (scaled by 0.01) breaks ties, prioritising clinically significant symptoms.

In [ ]:
ENSEMBLE_WEIGHTS = {"fuzzy": 0.30, "tfidf": 0.20, "sbert": 0.50}
ENSEMBLE_THRESHOLD = 0.30
SEVERITY_BONUS_SCALE = 0.01


def ensemble_match(query: str, top_k: int = 5):
    f_res = {r['canonical']: r['score'] for r in fuzzy_match(query, top_k=top_k * 2)}
    t_res = {r['canonical']: r['score'] for r in tfidf_match(query, top_k=top_k * 2)}
    s_res = {r['canonical']: r['score'] for r in sbert_match(query, top_k=top_k * 2)}
    all_cands = set(f_res) | set(t_res) | set(s_res)
    scored = []
    for cand in all_cands:
        f = f_res.get(cand, 0.0)
        t = t_res.get(cand, 0.0)
        s = s_res.get(cand, 0.0)
        ens = ENSEMBLE_WEIGHTS["fuzzy"] * f + ENSEMBLE_WEIGHTS["tfidf"] * t + ENSEMBLE_WEIGHTS["sbert"] * s
        sev = severity_lookup.get(cand, 3)
        idx = canonical_symptoms.index(cand)
        scored.append({
            "canonical": cand,
            "display": symptom_display[idx],
            "score_fuzzy": round(f, 4),
            "score_tfidf": round(t, 4),
            "score_sbert": round(s, 4),
            "ensemble_score": round(ens + SEVERITY_BONUS_SCALE * sev, 4),
            "severity": sev,
        })
    scored.sort(key=lambda x: x['ensemble_score'], reverse=True)
    return [r for r in scored[:top_k] if r['ensemble_score'] >= ENSEMBLE_THRESHOLD]


print("Ensemble Fusion Results:\n")
for q in test_queries:
    matches = ensemble_match(q)
    print(f"Query: {repr(q)}")
    if matches:
        cols = ["display", "score_fuzzy", "score_tfidf", "score_sbert", "ensemble_score", "severity"]
        print(pd.DataFrame(matches)[cols].to_string(index=False))
    else:
        print("  No confident match found.")
    print()

---

## 8. End-to-End Inference Pipeline

The `SymptomMatcher` class wraps all three strategies and exposes a single `parse_to_feature_vector` method. Given a free-text complaint string, the method:

1. Segments the input on common delimiters (commas, `and`, semicolons) to extract individual symptom phrases.
2. Runs the ensemble matcher on each phrase independently.
3. Selects the top candidate per phrase if the ensemble score exceeds the configured threshold.
4. Returns a 132-dimensional binary NumPy array compatible with any downstream classifier trained on `Training.csv`.

In [ ]:
class SymptomMatcher:
    def __init__(
        self, canonical_list, display_list, severity_map, tfidf_vec, tfidf_mat,
        sbert_encoder, corpus_embs, fuzzy_threshold=0.60, tfidf_threshold=0.20,
        sbert_threshold=0.45, ensemble_threshold=0.30, ensemble_weights=None,
    ):
        self.canonical_list = canonical_list
        self.display_list = display_list
        self.severity_map = severity_map
        self.tfidf_vec = tfidf_vec
        self.tfidf_mat = tfidf_mat
        self.sbert_encoder = sbert_encoder
        self.corpus_embs = corpus_embs
        self.fuzzy_threshold = fuzzy_threshold
        self.tfidf_threshold = tfidf_threshold
        self.sbert_threshold = sbert_threshold
        self.ensemble_threshold = ensemble_threshold
        self.weights = ensemble_weights or {"fuzzy": 0.30, "tfidf": 0.20, "sbert": 0.50}
        self._norm_symptom = [normalize_query(s.replace("_", " ")) for s in canonical_list]

    def _fuzzy(self, q_norm, top_k):
        wr = rf_process.extract(q_norm, self._norm_symptom, scorer=rf_fuzz.WRatio, limit=top_k * 2)
        ts = rf_process.extract(q_norm, self._norm_symptom, scorer=rf_fuzz.token_set_ratio, limit=top_k * 2)
        sc = {}
        for _, s, i in wr:
            sc[i] = max(sc.get(i, 0), s / 100.0)
        for _, s, i in ts:
            sc[i] = max(sc.get(i, 0), s / 100.0)
        return {self.canonical_list[i]: v for i, v in sc.items() if v >= self.fuzzy_threshold}

    def _tfidf(self, q_norm, top_k):
        q_vec = self.tfidf_vec.transform([q_norm])
        sims = cosine_similarity(q_vec, self.tfidf_mat).flatten()
        top_idx = sims.argsort()[::-1][:top_k]
        return {self.canonical_list[i]: float(sims[i]) for i in top_idx if sims[i] >= self.tfidf_threshold}

    def _sbert(self, query, top_k):
        q_emb = self.sbert_encoder.encode([query], normalize_embeddings=True)
        sims = (q_emb @ self.corpus_embs.T).flatten()
        top_idx = sims.argsort()[::-1][:top_k]
        return {self.canonical_list[i]: float(sims[i]) for i in top_idx if sims[i] >= self.sbert_threshold}

    def match_phrase(self, phrase: str, top_k: int = 3):
        q_norm = normalize_query(phrase)
        f_sc = self._fuzzy(q_norm, top_k)
        t_sc = self._tfidf(q_norm, top_k)
        s_sc = self._sbert(phrase, top_k)
        candidates = set(f_sc) | set(t_sc) | set(s_sc)
        results = []
        for cand in candidates:
            f = f_sc.get(cand, 0.0)
            t = t_sc.get(cand, 0.0)
            s = s_sc.get(cand, 0.0)
            ens = self.weights["fuzzy"] * f + self.weights["tfidf"] * t + self.weights["sbert"] * s
            sev = self.severity_map.get(cand, 3)
            results.append({"canonical": cand, "ensemble_score": ens + 0.01 * sev})
        results.sort(key=lambda x: x['ensemble_score'], reverse=True)
        return [r for r in results[:top_k] if r['ensemble_score'] >= self.ensemble_threshold]

    def parse_to_feature_vector(self, free_text: str):
        phrases = re.split(r"[,;]|\band\b", free_text, flags=re.IGNORECASE)
        phrases = [p.strip() for p in phrases if p.strip()]
        feature_vector = np.zeros(len(self.canonical_list), dtype=np.int8)
        matched_symptoms = []
        for phrase in phrases:
            top_matches = self.match_phrase(phrase, top_k=1)
            if top_matches:
                best = top_matches[0]['canonical']
                idx = self.canonical_list.index(best)
                feature_vector[idx] = 1
                matched_symptoms.append((phrase, best, top_matches[0]['ensemble_score']))
        return feature_vector, matched_symptoms


matcher = SymptomMatcher(
    canonical_list=canonical_symptoms,
    display_list=symptom_display,
    severity_map=severity_lookup,
    tfidf_vec=tfidf_vectorizer,
    tfidf_mat=tfidf_matrix,
    sbert_encoder=sbert_model,
    corpus_embs=corpus_embeddings,
)
print("SymptomMatcher initialized.")

In [ ]:
demo_inputs = [
    "I have been vomiting, stomch ache, and feel very tired",
    "skin is itchy with red spots and peeling",
    "difficulty breathing, chest tightness and heart pounding fast",
    "blured vision, severe headche and spinning sensation",
    "loss of appetite, dark yellowish urine and yellowing of eyes",
]

for user_input in demo_inputs:
    vec, matched = matcher.parse_to_feature_vector(user_input)
    print(f"Input : {repr(user_input)}")
    total = int(vec.sum())
    print(f"Active symptoms ({total}):")
    for phrase, sym, score in matched:
        print(f"  {repr(phrase)} -> {repr(sym)} (ensemble={score:.3f})")
    print(f"Active indices: {np.where(vec == 1)[0].tolist()}")
    print()

---

## 9. Evaluation Against Ground-Truth Pairs

A synthetic evaluation dataset is constructed from the canonical symptom vocabulary. Each canonical symptom is perturbed with controlled character-level noise (random keyboard-proximity typos and word reordering). The matcher is scored on whether it recovers the correct canonical feature.

Metrics reported:

- **Hit@1**: The top-ranked result is the correct canonical symptom.
- **Hit@3**: The correct canonical symptom appears in the top 3 results.
- **Mean Reciprocal Rank (MRR)**: Average of the reciprocal rank of the first correct result.
- **Micro Precision, Recall, F1** on binary feature vector recovery across 200 sampled training records.

In [ ]:
TYPO_MAP = {"a": "q", "e": "r", "i": "u", "o": "p", "s": "z",
            "n": "m", "t": "y", "g": "h", "d": "f", "l": "k"}


def inject_typo(word: str, prob: float = 0.25) -> str:
    if len(word) < 4 or np.random.rand() > prob:
        return word
    idx = np.random.randint(1, len(word) - 1)
    char = word[idx]
    return word[:idx] + TYPO_MAP.get(char, char) + word[idx + 1:]


def perturb_symptom(canonical: str) -> str:
    words = canonical.replace("_", " ").split()
    words = [inject_typo(w) for w in words]
    if np.random.rand() > 0.5 and len(words) > 1:
        np.random.shuffle(words)
    return " ".join(words)


np.random.seed(42)
eval_records = [{"ground_truth": s, "query": perturb_symptom(s)} for s in canonical_symptoms]
df_eval = pd.DataFrame(eval_records)
print(f"Evaluation pairs: {len(df_eval)}")
print(df_eval.head(10).to_string(index=False))

In [ ]:
strategy_configs = {
    "Fuzzy (RapidFuzz)": lambda q: [r["canonical"] for r in fuzzy_match(q, top_k=3)],
    "TF-IDF Cosine": lambda q: [r["canonical"] for r in tfidf_match(q, top_k=3)],
    "SBERT Semantic": lambda q: [r["canonical"] for r in sbert_match(q, top_k=3)],
    "Ensemble": lambda q: [r["canonical"] for r in ensemble_match(q, top_k=3)],
}


def evaluate_strategy(name, retrieve_fn, df):
    hit1, hit3, rr_list = 0, 0, []
    for _, row in df.iterrows():
        gt = row["ground_truth"]
        results = retrieve_fn(row["query"])
        if results and results[0] == gt:
            hit1 += 1
        if gt in results:
            hit3 += 1
            rr_list.append(1.0 / (results.index(gt) + 1))
        else:
            rr_list.append(0.0)
    n = len(df)
    return {"Strategy": name, "Hit@1": round(hit1 / n, 4), "Hit@3": round(hit3 / n, 4), "MRR": round(np.mean(rr_list), 4)}


eval_results = []
for name, fn in strategy_configs.items():
    print(f"Evaluating: {name}...")
    res = evaluate_strategy(name, fn, df_eval)
    eval_results.append(res)
    print(f"  Hit@1={res['Hit@1']}, Hit@3={res['Hit@3']}, MRR={res['MRR']}")

df_results = pd.DataFrame(eval_results)
print("\nEvaluation Summary:")
print(df_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_results))
width = 0.26
bars1 = ax.bar(x - width, df_results["Hit@1"], width, label="Hit@1", color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x, df_results["Hit@3"], width, label="Hit@3", color="#55A868", edgecolor="white")
bars3 = ax.bar(x + width, df_results["MRR"], width, label="MRR", color="#C44E52", edgecolor="white")
ax.set_xticks(x)
ax.set_xticklabels(df_results["Strategy"], fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_title("Retrieval Strategy Comparison: Hit@1, Hit@3, MRR", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)
for bar_group in [bars1, bars2, bars3]:
    for bar in bar_group:
        h = bar.get_height()
        ax.annotate(f"{h:.2f}", xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "02_strategy_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
print("Full-Record Binary Feature Vector Evaluation (Ensemble):\n")

sample_records = df_train.sample(200, random_state=42)
feature_cols = canonical_symptoms
all_y_true, all_y_pred = [], []

for _, row in sample_records.iterrows():
    active_syms = [feature_cols[i] for i, v in enumerate(row[feature_cols]) if v == 1]
    if not active_syms:
        continue
    free_text = ", ".join([s.replace("_", " ") for s in active_syms])
    pred_vec, _ = matcher.parse_to_feature_vector(free_text)
    true_vec = row[feature_cols].values.astype(np.int8)
    all_y_true.append(true_vec)
    all_y_pred.append(pred_vec)

Y_true = np.vstack(all_y_true)
Y_pred = np.vstack(all_y_pred)

overall_precision = precision_score(Y_true, Y_pred, average="micro", zero_division=0)
overall_recall = recall_score(Y_true, Y_pred, average="micro", zero_division=0)
overall_f1 = f1_score(Y_true, Y_pred, average="micro", zero_division=0)
overall_acc = accuracy_score(Y_true.flatten(), Y_pred.flatten())

print(f"Micro Precision       : {overall_precision:.4f}")
print(f"Micro Recall          : {overall_recall:.4f}")
print(f"Micro F1-Score        : {overall_f1:.4f}")
print(f"Element-wise Accuracy : {overall_acc:.4f}")

In [ ]:
per_symptom_f1 = f1_score(Y_true, Y_pred, average=None, zero_division=0)
support = Y_true.sum(axis=0)

df_per_symptom = pd.DataFrame({
    "symptom": feature_cols,
    "f1_score": per_symptom_f1,
    "support": support,
}).query("support > 0").sort_values("f1_score")

fig, ax = plt.subplots(figsize=(10, 12))
colors = ["#d73027" if f < 0.5 else "#4575b4" if f >= 0.8 else "#fee090"
          for f in df_per_symptom["f1_score"]]
ax.barh(df_per_symptom["symptom"], df_per_symptom["f1_score"],
        color=colors, edgecolor="white", height=0.7)
ax.axvline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.7, label="F1 = 0.50")
ax.axvline(0.8, color="blue", linestyle="--", linewidth=1, alpha=0.7, label="F1 = 0.80")
ax.set_xlabel("F1-Score", fontsize=11)
ax.set_title("Per-Symptom F1-Score (Ensemble Matcher)", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.set_xlim(0, 1.05)
plt.tight_layout()
plt.savefig(os.path.join(ARTIFACT_DIR, "03_per_symptom_f1.png"), dpi=150, bbox_inches="tight")
plt.show()

---

## 10. Artifact Persistence

All model components required for inference are serialized to `Model/symptom_matcher_artifacts/`.

| Artifact | File | Description |
|---|---|---|
| Canonical symptom vocabulary | `canonical_symptoms.json` | Ordered list of 132 feature column names |
| Severity weight lookup | `severity_lookup.json` | Dict mapping symptom to clinical severity weight |
| TF-IDF vectorizer | `tfidf_vectorizer.joblib` | Fitted TfidfVectorizer instance |
| TF-IDF matrix | `tfidf_corpus_matrix.npy` | Precomputed matrix (dense serialized) |
| SBERT corpus embeddings | `sbert_corpus_embeddings.npy` | 132 x 384 float32 dense embeddings |
| Evaluation results | `evaluation_results.csv` | Per-strategy metric table |
| Matcher configuration | `matcher_config.json` | Thresholds and ensemble weight settings |

In [ ]:
with open(os.path.join(ARTIFACT_DIR, "canonical_symptoms.json"), "w") as f:
    json.dump(canonical_symptoms, f, indent=2)

with open(os.path.join(ARTIFACT_DIR, "severity_lookup.json"), "w") as f:
    json.dump(severity_lookup, f, indent=2)

joblib.dump(tfidf_vectorizer, os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.joblib"))
np.save(os.path.join(ARTIFACT_DIR, "tfidf_corpus_matrix.npy"),
        tfidf_matrix.toarray().astype(np.float32))
df_results.to_csv(os.path.join(ARTIFACT_DIR, "evaluation_results.csv"), index=False)

matcher_config = {
    "sbert_model_name": SBERT_MODEL_NAME,
    "fuzzy_threshold": FUZZY_THRESHOLD,
    "tfidf_threshold": TFIDF_THRESHOLD,
    "sbert_threshold": SBERT_THRESHOLD,
    "ensemble_threshold": ENSEMBLE_THRESHOLD,
    "ensemble_weights": ENSEMBLE_WEIGHTS,
    "severity_bonus_scale": SEVERITY_BONUS_SCALE,
    "num_canonical_symptoms": len(canonical_symptoms),
}
with open(os.path.join(ARTIFACT_DIR, "matcher_config.json"), "w") as f:
    json.dump(matcher_config, f, indent=2)

print("All artifacts saved to:", os.path.abspath(ARTIFACT_DIR))
print("\nSaved files:")
for fname in sorted(os.listdir(ARTIFACT_DIR)):
    fpath = os.path.join(ARTIFACT_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:50s}  {size_kb:8.1f} KB")

In [ ]:
print("Verifying inference pipeline from persisted artifacts...\n")

loaded_canonical = json.load(open(os.path.join(ARTIFACT_DIR, "canonical_symptoms.json")))
loaded_severity = json.load(open(os.path.join(ARTIFACT_DIR, "severity_lookup.json")))
loaded_tfidf_vec = joblib.load(os.path.join(ARTIFACT_DIR, "tfidf_vectorizer.joblib"))
loaded_tfidf_mat = np.load(os.path.join(ARTIFACT_DIR, "tfidf_corpus_matrix.npy"))
loaded_corpus_embs = np.load(os.path.join(ARTIFACT_DIR, "sbert_corpus_embeddings.npy"))
loaded_config = json.load(open(os.path.join(ARTIFACT_DIR, "matcher_config.json")))
loaded_sbert = SentenceTransformer(loaded_config["sbert_model_name"])

loaded_matcher = SymptomMatcher(
    canonical_list=loaded_canonical,
    display_list=[s.replace("_", " ") for s in loaded_canonical],
    severity_map=loaded_severity,
    tfidf_vec=loaded_tfidf_vec,
    tfidf_mat=loaded_tfidf_mat,
    sbert_encoder=loaded_sbert,
    corpus_embs=loaded_corpus_embs,
    fuzzy_threshold=loaded_config["fuzzy_threshold"],
    tfidf_threshold=loaded_config["tfidf_threshold"],
    sbert_threshold=loaded_config["sbert_threshold"],
    ensemble_threshold=loaded_config["ensemble_threshold"],
    ensemble_weights=loaded_config["ensemble_weights"],
)

test_input = "I have joint pain, fatigue, and high fever with sweating"
vec, matched = loaded_matcher.parse_to_feature_vector(test_input)

print(f"Test input : {repr(test_input)}")
total = int(vec.sum())
print(f"Matched symptoms ({total}):")
for phrase, sym, score in matched:
    print(f"  {repr(phrase)} -> {repr(sym)} (ensemble={score:.3f})")
print(f"Feature vector length: {len(vec)}")
print("\nPipeline reload and verification completed successfully.")

---

## Summary

This notebook constructed and evaluated a three-strategy ensemble symptom matcher capable of parsing unstructured patient text into structured binary feature vectors. Key takeaways:

- **Fuzzy matching (RapidFuzz)** provides high-precision recovery of typo-perturbed exact symptom labels.
- **TF-IDF cosine retrieval** (char-level 2-4-gram) handles partial token overlap effectively and runs entirely offline without a neural network.
- **SBERT semantic search** (`all-MiniLM-L6-v2`) bridges vocabulary gaps by encoding contextual meaning, enabling robust matching for paraphrased or synonymous descriptions.
- **Ensemble fusion** consistently outperforms any single strategy across Hit@1, Hit@3, and MRR metrics.

The persisted artifacts in `Model/symptom_matcher_artifacts/` are ready to be loaded by any downstream classification pipeline that expects a 132-dimensional binary input vector aligned with `Training.csv`.